In [ ]:
from typing import NamedTuple, Self
from jaxtyping import Array, Float, Scalar
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import jax.scipy as jsp
import jax.random as jr

import numpy as np
import scipy as sp

jax.config.update("jax_enable_x64", True)
EPS = float(jnp.sqrt(jnp.finfo(float).eps))

# Berstein Polynomial

In [ ]:
class BersteinPolynomial(NamedTuple):
    a: Float[Array, "n+1"]  # Coefficients

    def bump_degree(self) -> Self:
        n = len(self.a) - 1
        c = jnp.arange(n + 2) / (n + 1)
        a = jnp.zeros(n + 2)
        a = a.at[1:].add(c[1:] * self.a)
        a = a.at[:-1].add((1 - c[:-1]) * self.a)
        return self._replace(a=a)

    def derivative(self) -> Self:
        n = len(self.a) - 1
        a = n * (self.a[1:] - self.a[:-1])
        return self._replace(a=a)

    def __call__(self, x: Float[Array, "m 1"]) -> Float[Array, "m"]:
        n = len(self.a) - 1
        g = jsp.stats.binom.pmf(jnp.arange(n + 1), n, x)
        return jnp.sum(self.a * g, axis=-1)

In [ ]:
def test_fn(x: Float[Array, "m 1"]) -> Float[Array, "m"]:
    return jnp.sinc(x * 2 * jnp.pi).squeeze()


def target_fn(f: BersteinPolynomial, n: int = 100000) -> Scalar:
    x = jnp.linspace(0, 1, n).reshape(-1, 1)
    return jnp.mean((f(x) - test_fn(x)) ** 2)


def plot_fns(
    fs: list[BersteinPolynomial] = [],
    background_fs: list[BersteinPolynomial] = [],
) -> None:
    x = jnp.linspace(0, 1, 1000).reshape(-1, 1)
    plt.plot(x, test_fn(x), label="target", color="black", linestyle="dashed")
    for i, f in enumerate(background_fs):
        plt.plot(x, f(x), color="gray", alpha=0.3)
    for i, f in enumerate(fs):
        plt.plot(x, f(x))
    plt.legend()
    plt.grid()
    plt.show()

# Fit GP

In [ ]:
class Gaussian(NamedTuple):
    mean: Float[Array, "n"]
    cov: Float[Array, "n n"]


class GaussianProcess(NamedTuple):
    # model parameters
    rho: Float[Array, "d"]  # lengthscale
    g: Scalar  # nugget
    nu: Scalar  # covariance scale
    b: Scalar  # trend

    # observed data
    observed_xs: Float[Array, "n d"]
    observed_ys: Float[Array, "n"]

    # cached covariance matrix of the observed data
    Koo: Float[Array, "n n"]

    @classmethod
    def fit(
        cls,
        xs: Float[Array, "n d"],
        ys: Float[Array, "n"],
        *,
        lengthscale_range: tuple[float, float] = (EPS, 100.0),
        nugget_range: tuple[float, float] = (EPS, 1e-3),
        max_iterations: int = 100,
    ):
        n, d = xs.shape

        # define the loss function for optimization
        def loglikelihood(
            rho: Scalar, g: Scalar, xs: Float[Array, "n d"], ys: Float[Array, "n"]
        ) -> tuple[Scalar, Scalar, Scalar]:
            # foward of kernel
            K = cls.kernel(rho, xs, xs)
            K = K + jnp.eye(len(ys)) * g

            # cholesky of K and compute logdet
            K_sqrt, is_lower = jsp.linalg.cho_factor(K)
            logdetK = 2.0 * jnp.sum(jnp.log(jnp.diag(K_sqrt)))

            # compute Ki_1=(K^-1 @ 1) and Ki_y=(K^-1 @ y)
            Ki_1, Ki_y = jsp.linalg.cho_solve(
                c_and_lower=(K_sqrt, is_lower),
                b=jnp.stack([jnp.ones_like(ys), ys], 1),
            ).T

            # compute optimal trend b and scale nu
            b = (Ki_1 * ys).sum() / Ki_1.sum()
            nu = jnp.dot((ys - b) / len(ys), (Ki_y - Ki_1 * b))

            # likelihood when marginalizing over trend and variance
            loglik = -0.5 * (len(ys) * jnp.log(nu) + logdetK)
            return loglik, b, nu

        @jax.jit
        @jax.value_and_grad
        def mle_loss(params: Float[Array, "d+1"]):
            rho, g = params[:-1], params[-1]
            loglik, b, nu = loglikelihood(rho, g, xs, ys)
            return -loglik

        # initialization
        nugget = min(0.1, nugget_range[1])
        lengthscale = 0.9 * lengthscale_range[1] + 0.1 * lengthscale_range[0]
        init_params = jnp.array([lengthscale for _ in range(d)] + [nugget])


        # run optimization
        result = sp.optimize.minimize(
            fun=mle_loss,
            x0=init_params,
            jac=True,
            method="L-BFGS-B",
            bounds=[lengthscale_range] * d + [nugget_range],
            options=dict(maxiter=max_iterations, ftol=EPS, gtol=0),
        )

        # extract the optimal parameters and infer the rest
        rho = jnp.array(result.x[:-1])
        g = jnp.array(result.x[-1])
        llk, b, nu = loglikelihood(rho, g, xs, ys)

        # precompute the covariance matrix of the observed data for faster predictions
        Koo = nu * (cls.kernel(rho, xs, xs) + g * jnp.eye(n))
        return cls(rho=rho, g=g, nu=nu, b=b, observed_xs=xs, observed_ys=ys, Koo=Koo)

    @staticmethod
    def kernel(
        rho: Float[Array, "d"], xs_1: Float[Array, "n d"], xs_2: Float[Array, "m d"]
    ) -> Float[Array, "n m"]:
        def dist_squared(x1: Float[Array, "d"], x2: Float[Array, "d"]) -> Scalar:
            return jnp.sum(jnp.square(x1 - x2) / rho)

        d2 = jax.vmap(jax.vmap(dist_squared, in_axes=(None, 0)), in_axes=(0, None))
        k = jnp.exp(-0.5 * d2(xs_1, xs_2))
        return k

    @jax.jit
    def predict(self, xs: Float[Array, "m d"]) -> Gaussian:
        n = len(self.observed_ys)

        # compute covariance matrices
        Kxx = self.nu * self.kernel(self.rho, xs, xs) 
        Kxo = self.nu * self.kernel(self.rho, xs, self.observed_xs)
        Koo = self.Koo

        # posterior mean and covariance
        mean = self.b + Kxo @ jnp.linalg.solve(Koo, self.observed_ys - self.b)
        cov = Kxx - Kxo @ jnp.linalg.solve(Koo, Kxo.T)

        # Add correction based on the trend estimation correlation
        Kbx = jnp.ones((1, n)) @ jnp.linalg.solve(Koo, Kxo.T)
        cov = cov + (1 - Kbx).T @ (1 - Kbx) / jnp.linalg.inv(Koo).sum()
        return Gaussian(mean=mean, cov=cov)

    @jax.jit
    def log_expected_improvement(self, x: Float[Array, "d"]) -> Scalar:
        # numerically stable version following https://arxiv.org/pdf/2310.20708:
        mu, cov = self.predict(x[None, :])
        mu = mu.squeeze()
        sigma = cov.squeeze() ** 0.5
        z = (self.observed_ys.min() - mu) / sigma

        # use lax.cond to avoid propagating NaNs in the gradients
        branch1 = lambda: jnp.log(z * jsp.stats.norm.cdf(z) + jsp.stats.norm.pdf(z))
        branch2a = lambda: -2 * jnp.log(-z)
        branch2b = lambda: jax.nn.log1mexp(
            -jnp.log(-z)
            - jsp.stats.norm.logsf(-z)
            - z**2 / 2
            - jnp.log(2 * jnp.pi) / 2.0
        )
        branch2 = lambda: (
            -(z**2) / 2
            - jnp.log(2 * jnp.pi) / 2
            + jax.lax.cond(z < -1 / jnp.sqrt(EPS), branch2a, branch2b)
        )
        ei = jnp.log(sigma) + jax.lax.cond(z > -1, branch1, branch2)
        return ei

# Expected Improvement

In [ ]:
class BFGS(NamedTuple):
    multi_starts: int
    raw_samples: int = 1024
    max_iterations: int = 100
    ftol: float = EPS
    gtol: float = 0.0

    def __call__(self, model: GaussianProcess, n: int) -> Float[Array, "d"]:
        # define the acquisition function and its gradient as a vect
        @jax.jit
        @jax.value_and_grad
        def loss(x: Float[Array, "n+1"]):
            return -model.log_expected_improvement(x)

        # sample initial guess for candidate points
        lhs_sampler = sp.stats.qmc.LatinHypercube(d=n+1, seed=np.random.mtrand._rand)
        xs = lhs_sampler.random(n=self.raw_samples)

        # keep the most promising initial guesses
        losses = np.array([loss(xi)[0] for xi in xs])
        xs = xs[np.argsort(losses)[: self.multi_starts]]

        # optimize each initial guess with L-BFGS
        results = [
            sp.optimize.minimize(
                fun=loss,
                x0=xi,
                jac=True,
                method="L-BFGS-B",
                bounds=[(-1.0, 1.0)] * (n + 1),
                options=dict(
                    maxiter=self.max_iterations,
                    ftol=self.ftol,
                    gtol=self.gtol,
                ),
            )
            for xi in xs
        ]

        # sort results and return best q
        losses = np.array([result.fun for result in results])
        xs = np.array([result.x for result in results])
        return xs[np.argmin(losses)]

In [ ]:
INITIAL_ACQUISITION = 10
INITIAL_DEGREE = 5
MAX_DEGREE = 10
ACQUISITION_EACH_DEGREE = 30

lhs_sampler = sp.stats.qmc.LatinHypercube(
    d=INITIAL_DEGREE + 1, seed=np.random.mtrand._rand
)
xs = lhs_sampler.random(n=INITIAL_ACQUISITION)
ys = jnp.array([target_fn(BersteinPolynomial(a=xi)) for xi in xs])
models = [GaussianProcess.fit(xs, ys)]

for degree in range(INITIAL_DEGREE, MAX_DEGREE + 1):
    print(f"Acquisition with {degree} basis points")
    acquisition = BFGS(multi_starts=16 * degree)

    for i in range(ACQUISITION_EACH_DEGREE):
        x = acquisition(models[-1], n=degree)
        y = target_fn(BersteinPolynomial(a=x))

        xs = jnp.concatenate([xs, x[None]])
        ys = jnp.concatenate([ys, y[None]])
        models.append(GaussianProcess.fit(xs, ys))

        best_idx = jnp.argmin(ys)
        print(f"Iteration {i+1}: current= {y:.8f}, best = {ys[best_idx]:.8f}")

    best_idx = jnp.argmin(ys)
    plot_fns(
        fs=[BersteinPolynomial(a=xs[best_idx])],
        background_fs=[BersteinPolynomial(a=xi) for xi in xs[-ACQUISITION_EACH_DEGREE:]],
    )

    if degree < MAX_DEGREE:
        xs = jnp.stack([BersteinPolynomial(a=xi).bump_degree().a for xi in xs])
        models.append(GaussianProcess.fit(xs, ys))

In [ ]:
plt.figure(figsize=(5, 3))
n0, dn = INITIAL_ACQUISITION, ACQUISITION_EACH_DEGREE 
plt.plot(range(0, n0), ys[:n0], "o", label="initial samples")
for i, degree in enumerate(range(INITIAL_DEGREE, MAX_DEGREE + 1)):
    y_deg = ys[n0 + i * dn : n0 + (i + 1) * dn]
    plt.plot(range(n0 + i * dn, n0 + (i + 1) * dn), y_deg, "o", label=f"acquired samples (degree={degree})")

plt.yscale("log")
plt.xlabel("Total Evaluations")
plt.ylabel("Target fn")
plt.title("Convergence of Bayesian Optimization")
plt.grid()
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()